### 1. Identify Unique Water Quality Measurement Sites

The following code defines a function `get_unique_water_quality_locations` that:
- Loads the `/content/station.csv` file into a pandas DataFrame.
- Filters the DataFrame to include only rows where 'Site_Type' or 'Station_Name' contains 'water quality' (case-insensitive).
- Selects relevant location columns ('Station_ID', 'Station_Name', 'LatitudeMeasure', 'LongitudeMeasure', 'Address', 'City', 'State').
- Removes duplicate locations to ensure each unique physical site is listed only once.

After defining the function, it calls it and displays the resulting unique locations.

In [8]:
import pandas as pd

def get_unique_water_quality_locations(file_path='/content/station.csv'):
    """
    Loads station data, filters for water quality measurement sites,
    and returns unique location information.

    Args:
        file_path (str): The path to the station CSV file.

    Returns:
        pandas.DataFrame: A DataFrame containing unique location details
                          of water quality measurement sites.
    """
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found. Please ensure it is uploaded or the path is correct.")
        return pd.DataFrame()

    # Convert column names to lowercase for case-insensitive matching
    df.columns = df.columns.str.lower()

    # Filter for water quality measurement sites
    condition_site_type = pd.Series([False] * len(df))
    condition_station_name = pd.Series([False] * len(df))

    if 'site_type' in df.columns:
        condition_site_type = df['site_type'].str.contains('water quality', case=False, na=False)
    else:
        print("Warning: 'site_type' column not found. Filtering will rely only on 'station_name' if available.")

    if 'station_name' in df.columns:
        condition_station_name = df['station_name'].str.contains('water quality', case=False, na=False)
    else:
        print("Warning: 'station_name' column not found. Filtering might not be effective.")

    water_quality_sites = df[condition_site_type | condition_station_name]

    # Select relevant location columns. Ensure these columns exist in the original DataFrame.
    # Using original case column names for consistency with `df_dummy` values provided in the context.
    original_cols = ['Station_ID', 'Station_Name', 'LatitudeMeasure', 'LongitudeMeasure', 'Address', 'City', 'State']
    existing_location_cols = [col for col in original_cols if col.lower() in df.columns]

    # Rename columns in water_quality_sites to match original case if needed
    water_quality_sites.columns = [col.replace('latitudemeasure', 'LatitudeMeasure').replace('longitudemeasure', 'LongitudeMeasure') for col in water_quality_sites.columns]
    water_quality_sites.columns = [col.replace('station_id', 'Station_ID').replace('station_name', 'Station_Name') for col in water_quality_sites.columns]
    water_quality_sites.columns = [col.replace('site_type', 'Site_Type').replace('address', 'Address').replace('city', 'City').replace('state', 'State') for col in water_quality_sites.columns]

    location_info = water_quality_sites[existing_location_cols]

    # Display unique location information (remove duplicates based on location coordinates)
    unique_locations = location_info.drop_duplicates(subset=['LatitudeMeasure', 'LongitudeMeasure'])

    return unique_locations

# Call the function to get and display unique water quality site locations
unique_water_quality_locations = get_unique_water_quality_locations()

if not unique_water_quality_locations.empty:
    print("\nUnique Water Quality Measurement Site Locations:")
    display(unique_water_quality_locations)
else:
    print("No water quality measurement sites found or an error occurred.")

No water quality measurement sites found or an error occurred.


### 2. Create an Interactive Map of All Station Locations

The following code will:
- Install the `folium` library if it's not already installed.
- Define a function `create_station_map` that loads all station data from `/content/station.csv`.
- Creates an interactive map using `folium`, centered on the average latitude and longitude of the stations.
- Adds a marker for each station with a popup displaying details like Station ID, Name, Site Type, and Address, and a tooltip showing the station name.

In [9]:
import sys
!{sys.executable} -m pip install folium

In [10]:
import pandas as pd
import folium

def create_station_map(file_path='/content/station.csv'):
    """
    Loads station data and creates an interactive map with markers
    for each station.

    Args:
        file_path (str): The path to the station CSV file.

    Returns:
        folium.Map: An interactive folium map.
    """
    try:
        df = pd.read_csv(file_path)
        print(f"\nData loaded from '{file_path}':")
        display(df.head())
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found. Please ensure it is uploaded or the path is correct.")
        return None

    # Ensure latitude and longitude columns exist and are numeric
    if 'LatitudeMeasure' not in df.columns or 'LongitudeMeasure' not in df.columns:
        print("Error: 'LatitudeMeasure' or 'LongitudeMeasure' column not found in the CSV.")
        return None

    df['LatitudeMeasure'] = pd.to_numeric(df['LatitudeMeasure'], errors='coerce')
    df['LongitudeMeasure'] = pd.to_numeric(df['LongitudeMeasure'], errors='coerce')
    df.dropna(subset=['LatitudeMeasure', 'LongitudeMeasure'], inplace=True)

    if df.empty:
        print("No valid latitude/longitude data to plot.")
        return None

    # Calculate the center of the map
    map_center = [df['LatitudeMeasure'].mean(), df['LongitudeMeasure'].mean()]

    # Create a Folium map object
    station_map = folium.Map(location=map_center, zoom_start=10)

    # Add markers for each station
    for index, row in df.iterrows():
        popup_html = (f"<b>Station ID:</b> {row.get('Station_ID', 'N/A')}<br>" \
                      f"<b>Name:</b> {row.get('Station_Name', 'N/A')}<br>" \
                      f"<b>Site Type:</b> {row.get('Site_Type', 'N/A')}<br>" \
                      f"<b>Address:</b> {row.get('Address', 'N/A')}, {row.get('City', 'N/A')}, {row.get('State', 'N/A')}")
        folium.Marker(
            location=[row['LatitudeMeasure'], row['LongitudeMeasure']],
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=row.get('Station_Name', 'Station')
        ).add_to(station_map)

    return station_map

# Call the function to create and display the map
station_map = create_station_map()

if station_map:
    display(station_map)
else:
    print("Could not generate map.")


Data loaded from '/content/station.csv':


,OrganizationIdentifier,OrganizationFormalName,MonitoringLocationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,MonitoringLocationDescriptionText,HUCEightDigitCode,DrainageAreaMeasure/MeasureValue,DrainageAreaMeasure/MeasureUnitCode,ContributingDrainageAreaMeasure/MeasureValue,...,AquiferName,LocalAqfrName,FormationTypeText,AquiferTypeName,ConstructionDateText,WellDepthMeasure/MeasureValue,WellDepthMeasure/MeasureUnitCode,WellHoleDepthMeasure/MeasureValue,WellHoleDepthMeasure/MeasureUnitCode,ProviderName
0,USGS-KY,USGS Kentucky Water Science Center,USGS-03254520,"LICKING RIVER AT HWY 536 NEAR ALEXANDRIA, KY",Stream,NaN,5100101,3593.0,sq mi,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
1,USGS-KY,USGS Kentucky Water Science Center,USGS-03290500,"KENTUCKY RIVER AT LOCK 2 AT LOCKPORT, KY",Stream,NaN,5100205,6180.0,sq mi,5984.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
2,USGS-KY,USGS Kentucky Water Science Center,USGS-03302058,"Salt River at Main Range Rd nr West Point, KY",Stream,NaN,5140102,2790.0,sq mi,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
3,11NPSWRD_WQX,National Park Service Water Resources Division,11NPSWRD_WQX-ABLI_HSSS,Howell Spring,Spring,Howell Spring is located just outside the park...,5110001,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STORET
4,11NPSWRD_WQX,National Park Service Water Resources Division,11NPSWRD_WQX-ABLI_KCKC,Knob Creek,River/Stream,"The park's new lands cross US31E, and for a sh...",5140103,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STORET
